# 🧪 Lab 13 — Fine-tuning & Open-Weight Models dari Nol

> Pendamping materi Bab 13. Kamu menjalankan **siklus fine-tuning lengkap
> tanpa GPU**: dataset SFT → split & oversampling → matematika LoRA →
> gradient check → kuantisasi → trainer SGD + early stopping → keputusan
> biaya API vs lokal → evaluasi sebelum/sesudah + kebijakan OOD.

"Model" yang di-fine-tune di lab ini **sengaja tiny** (klasifier bag-of-words,
17 fitur, SGD murni-Python, < 0,1 detik/run) supaya: (1) angka **terkunci dan
reproducible** di komputer siapa pun — syarat evaluasi yang adil; (2) kamu bisa
bereksperimen puluhan kali, bukan sekali semalam; (3) **matematikanya sama**
dengan SFT LoRA 7B di Unsloth/Colab — yang berubah saat skala naik hanyalah
jumlah param, VRAM, dan waktu, bukan bentuk loop-nya.

| Bagian | Topik | Waktu (menit) |
|---|---|---|
| 0 | Setup: dataset, tokenizer, model tiny | 5 |
| 1 | Format SFT & JSONL round-trip | 15 |
| 2 | Split & oversampling deterministik | 15 |
| 3 | Matematika LoRA (ΔW = B@A) | 20 |
| 4 | Gradient check | 10 |
| 5 | Kuantisasi & kompresi | 15 |
| 6 | Trainer SGD + early stopping | 25 |
| 7 | Biaya API vs lokal | 15 |
| 8 | Evaluasi sebelum/sesudah + OOD | 20 |
| 9 | Ringkasan & pertanyaan analisis | 10 |

Setiap latihan punya cell **✅ Cek** — jalankan; kalau ✅ berarti pemahamanmu benar.
Semua angka terkunci SEED 13.


## Bagian 0 — Setup: Dataset, Tokenizer, Model Tiny

`project-ftkit/ftkit/` menyediakan (GIVEN, jangan diubah):

- `data.py` — 24 ulasan e-commerce (16 train + 8 val) + 1 contoh OOD,
  FORMAT_SFT (8 baris, sengaja timpang 5/3), tarif API, varian kuantisasi;
- `tokenizer.py` — tokenizer mini deterministik (vocab dari train, OOV → `<unk>`,
  flag negasi) → fitur 17-dim;
- `model.py` — klasifier tiny: `logit = w·x + b` → sigmoid → loss BCE + gradien
  analitik `dL/dw = (p - y)·x`.


In [ ]:
import json
import math
import sys
import tempfile
from pathlib import Path

# Cari folder project-ftkit (GIVEN layer ada di sana)
_akar = Path.cwd()
while _akar != _akar.parent and not (_akar / 'project-ftkit' / 'ftkit').is_dir():
    _akar = _akar.parent
sys.path.insert(0, str(_akar / 'project-ftkit'))

from ftkit.data import (DATA_OOD, DATA_SENTIMEN, FORMAT_SFT, HARGA, LABEL_NEG,
                        LABEL_POS, SEED, TEMPLATE_INSTRUKSI, VARIAN_KUANTISASI)
from ftkit.model import akurasi, forward, gradien, inisialisasi, loss_bce, loss_batch
from ftkit.tokenizer import TokenizerMini, fitur

print('GIVEN layer dari:', _akar / 'project-ftkit')
print('SEED:', SEED)
print('dataset      :', len(DATA_SENTIMEN), 'contoh —',
      {s: sum(1 for d in DATA_SENTIMEN if d['split'] == s) for s in ('train', 'val')})
print('FORMAT_SFT   :', len(FORMAT_SFT), 'baris —',
      {r: sum(1 for b in FORMAT_SFT if b['respons'] == r) for r in (LABEL_POS, LABEL_NEG)})
print('OOD          :', repr(DATA_OOD['teks']))
print('tarif (USD/1jt token):', HARGA)


In [ ]:
tok = TokenizerMini()
w0, b0 = inisialisasi()
data = {'train': [(fitur(d['teks'], tok), 1.0 if d['label'] == LABEL_POS else 0.0)
                  for d in DATA_SENTIMEN if d['split'] == 'train'],
        'val':   [(fitur(d['teks'], tok), 1.0 if d['label'] == LABEL_POS else 0.0)
                  for d in DATA_SENTIMEN if d['split'] == 'val']}

print('vocab        :', tok.ukuran_vocab(), 'token')
print('dim fitur    :', len(data['train'][0][0]), '(16 mean one-hot + 1 flag negasi)')
print('loss awal    :', round(loss_batch(data['train'], w0, b0), 4), '(terkunci 0.6882)')
x_negasi = fitur('bagus tidak', tok)
print('fitur[16] (flag negasi) untuk \'bagus tidak\' :', x_negasi[16])
assert len(data['train']) == 16 and len(data['val']) == 8
assert abs(loss_batch(data['train'], w0, b0) - 0.6882) < 1e-3
print('✅ Setup benar — model tiny siap dilatih.')


## Bagian 1 — Format SFT & JSONL Round-trip

Dataset SFT = JSONL ala Alpaca: satu objek per baris dengan kunci `instruksi`,
`input`, `respons`. **Format prompt terkunci** (diuji): instruksi + baris kosong
+ `Input: <input>` + baris kosong + `Jawaban:`. Format yang bergoyang = model
bingung saat inference — konsistensi itu bagian dari data engineering.


In [ ]:
def ke_contoh_sft(baris):
    """Dict {instruksi, input, respons} → (teks_prompt, respons).

    Format prompt TERKUNCI: instruksi + '\n\n' + 'Input: <input>' +
    '\n\n' + 'Jawaban:'.
    """
    # TODO
    raise NotImplementedError


def tulis_jsonl(path, baris_list):
    """List of dict → JSONL: satu objek per baris, ensure_ascii=False."""
    # TODO
    raise NotImplementedError


def muat_jsonl(path):
    """Baca JSONL → list of dict; baris kosong dilewati."""
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 1.1 — format prompt terkunci
prompt, resp = ke_contoh_sft(FORMAT_SFT[0])
print(prompt)
print('→ respons:', resp)
assert prompt == ('Klasifikasikan sentimen ulasan berikut. Jawab HANYA satu kata:'
                  ' positif atau negatif.\n\nInput: Barang bagus, pengiriman cepat\n\nJawaban:'), 'format prompt harus persis seperti kontrak'
assert resp == 'positif'
print('✅ Latihan 1.1 benar! Format = instruksi + Input: + Jawaban: (baris kosong pemisah).')


In [ ]:
# ✅ Cek latihan 1.2 — round-trip JSONL utuh
tmp = Path(tempfile.mkdtemp()) / 'sft.jsonl'
tulis_jsonl(tmp, FORMAT_SFT)
print('isi file (2 baris pertama):')
print(''.join(tmp.read_text(encoding='utf-8').splitlines(True) [:2]), end='')
balik = muat_jsonl(tmp)
assert balik == FORMAT_SFT, 'round-trip harus utuh: tulis → muat = data asli'
assert len(tmp.read_text(encoding='utf-8').strip().splitlines()) == len(FORMAT_SFT)

# baris kosong di tengah harus dilewati, bukan error
tmp2 = Path(tempfile.mkdtemp()) / 'bolong.jsonl'
tmp2.write_text(json.dumps(FORMAT_SFT[0], ensure_ascii=False) + '\n\n'
                + json.dumps(FORMAT_SFT[1], ensure_ascii=False) + '\n', encoding='utf-8')
assert muat_jsonl(tmp2) == [FORMAT_SFT[0], FORMAT_SFT[1]]
print('✅ Latihan 1.2 benar! JSONL round-trip utuh; baris kosong dilewati.')


## Bagian 2 — Split & Oversampling Deterministik

Dua keputusan dataset yang sering salah: **split** dan **ketidakseimbangan**.
Keduanya harus deterministik — split yang berubah tiap run = evaluasi yang tak
bisa dibandingkan. Oversampling harus **mempertahankan urutan asli**; menggrup
per label dulu mengubah distribusi batch.


In [ ]:
def split_sft(baris_list, rasio_val=0.25, seed=13):
    """Bagi deterministik interleaved: contoh ke-i → val bila
    (i*7 + seed) % 10 < rasio_val*10. Return (train, val).

    Interleaved (bukan blok) supaya kedua split merepresentasikan seluruh data.
    """
    # TODO
    raise NotImplementedError


def oversample(baris_list, per_label=4):
    """Samakan jumlah contoh per label respons dengan duplikasi deterministik.

    Urutan ASLI dipertahankan; minoritas diduplikasi berulang (dari awal lagi)
    sampai kuota terpenuhi.
    """
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 2.1 — split terkunci (5, 3)
train_s, val_s = split_sft(FORMAT_SFT, 0.25)
print('split   :', (len(train_s), len(val_s)), '| val idx:',
      [i for i in range(len(FORMAT_SFT)) if (i * 7 + 13) % 10 < 2.5])
assert (len(train_s), len(val_s)) == (5, 3)
asli = sorted(json.dumps(b, sort_keys=True, ensure_ascii=False) for b in FORMAT_SFT)
hasil = sorted(json.dumps(b, sort_keys=True, ensure_ascii=False) for b in train_s + val_s)
assert hasil == asli, 'split harus mempertahankan SEMUA contoh tanpa diubah'
# catatan: urutan boleh berubah — split interleaved, bukan blok

# stabilitas: dua panggilan = hasil sama (deterministik)
t2, v2 = split_sft(FORMAT_SFT, 0.25)
assert (t2, v2) == (train_s, val_s)
print('✅ Latihan 2.1 benar! Split interleaved deterministik: (5, 3).')


In [ ]:
# ✅ Cek latihan 2.2 — oversample 6/6, urutan asli
ov = oversample(FORMAT_SFT, per_label=6)
print('oversample per_label=6 →', len(ov), 'baris —',
      {r: sum(1 for b in ov if b['respons'] == r) for r in (LABEL_POS, LABEL_NEG)})
assert len(ov) == 12
assert sum(1 for b in ov if b['respons'] == LABEL_POS) == 6
assert sum(1 for b in ov if b['respons'] == LABEL_NEG) == 6
# 8 baris pertama = urutan asli UTUH (ini yang sering salah!)
assert ov[:8] == list(FORMAT_SFT), 'urutan asli harus dipertahankan'
# duplikasi: minoritas negatif diduplikasi berulang dari awal daftarnya
print('duplikasi ke-1:', ov[8]['input'], '| ke-2:', ov[9]['input'])
assert ov[8]['input'] == 'Barang bagus, pengiriman cepat'   # dup positif (5→6)
assert ov[9]['input'] == 'Kualitas jelek, saya menyesal'    # dup negatif (3→6)
assert ov[10]['input'] == 'Aplikasinya ribet dan lambat'

# fungsi murni: input tidak berubah
sebelum = [dict(b) for b in FORMAT_SFT]
oversample(FORMAT_SFT, per_label=6)
assert FORMAT_SFT == sebelum
print('✅ Latihan 2.2 benar! 12 baris (6/6), urutan asli utuh, input tidak berubah.')


## Bagian 3 — Matematika LoRA (ΔW = B@A)

Full fine-tune memperbarui semua bobot. LoRA membekukan base dan melatih
adapter rank-rendah: `ΔW = B @ A` dengan `A: rank × dim_in`, `B: dim_out × rank`.

**B diinisialisasi NOL** → ΔW awal = 0 → saat adapter dipasang model berperilaku
persis base model. Menginisialisasi A **dan** B acak = model rusak sebelum
training mulai.

Param terlatih = `rank × (dim_in + dim_out)` — BUKAN rank produk penuh.


In [ ]:
def buat_lora(dim_in, dim_out, rank, seed=13):
    """Adapter LoRA: A (rank × dim_in) = 0.01*((i+j)%5 - 2);
    B (dim_out × rank) = NOL → ΔW awal 0 (model mulai utuh!).
    Return dict {'A', 'B', 'rank', 'dim_in', 'dim_out'}."""
    # TODO
    raise NotImplementedError


def delta_w(adapter):
    """ΔW = B @ A (bentuk dim_out × dim_in)."""
    # TODO
    raise NotImplementedError


def hitung_param(adapter):
    """Jumlah param terlatih = rank*(dim_in + dim_out)."""
    # TODO
    raise NotImplementedError


def rasio_param(dim_in, dim_out, rank):
    """hitung_param / (dim_in*dim_out)."""
    # TODO
    raise NotImplementedError


def kontribusi_adapter(x, adapter):
    """Kontribusi adapter ke logit 1-output: (B @ (A @ x))[0]."""
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 3.1 — bentuk, ΔW awal 0, param
ad = buat_lora(4, 3, 2)
print('A:', len(ad['A']), 'x', len(ad['A'][0]), '| B:', len(ad['B']), 'x', len(ad['B'][0]))
assert len(ad['A']) == 2 and len(ad['A'][0]) == 4   # rank × dim_in
assert len(ad['B']) == 3 and len(ad['B'][0]) == 2   # dim_out × rank
assert all(v == 0.0 for row in delta_w(ad) for v in row), 'ΔW awal harus 0 (B nol)'
assert hitung_param(ad) == 14                        # 2*(4+3)
print('param(4,3,r2) =', hitung_param(ad), '| param(4096², r16) =',
      hitung_param(buat_lora(4096, 4096, 16)))
print('✅ Latihan 3.1 benar! B nol → model mulai UTUH; param = rank×(in+out).')


In [ ]:
# ✅ Cek latihan 3.2 — matematika ΔW = B@A & rasio 0.78%
ad2 = buat_lora(2, 2, 1)
ad2['A'] = [[1.0, 2.0]]
ad2['B'] = [[3.0], [4.0]]
d = delta_w(ad2)
print('ΔW =', d)
assert d == [[3.0, 6.0], [4.0, 8.0]]   # B@A per elemen

rasio = rasio_param(4096, 4096, 16)
print('rasio 4096² r16 :', rasio, '=', round(rasio * 100, 4), '%')
assert abs(rasio - 0.0078125) < 1e-9
assert abs(rasio_param(4096, 4096, 8) - 0.00390625) < 1e-9   # rank 8 = setengahnya

ad3 = buat_lora(2, 1, 1); ad3['A'] = [[1.0, 2.0]]; ad3['B'] = [[3.0]]
k = kontribusi_adapter([1.0, 2.0], ad3)
print('kontribusi (B@(A@x))[0] untuk x=[1,2]:', k, '(=3*(1*1+2*2))')
assert abs(k - 15.0) < 1e-9
ad4 = buat_lora(2, 1, 1)
assert kontribusi_adapter([1.0, 2.0], ad4) == 0.0      # adapter nol tak mengubah
print('✅ Latihan 3.2 benar! 131.072 param dari 16.7 jt = 0.78% — itulah hemat LoRA.')


## Bagian 4 — Gradient Check: Percaya Gradien Sebelum Percaya Training

Gradien analitik `dL/dw = (p-y)·x` harus cocok dengan gradien numerik
(central difference, h=1e-5) SEBELUM loop training dipercaya. Kalau meleset,
training \"jalan\" tapi tidak konvergen — bug yang mahal dicari di model besar.


In [ ]:
def gradient_check(tol=1e-6):
    """Selisih maksimum gradien analitik vs numerik (central diff h=1e-5).

    Pakai inisialisasi(), forward(), gradien(), loss_batch() dari ftkit.model.
    Return selisih absolut maksimum; wajar < 1e-6.
    """
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 4.1 — gradien analitik = numerik
maks = gradient_check()
print('max err:', maks)
assert maks < 1e-6, f'gradien analitik meleset: {maks}'
print('✅ Latihan 4.1 benar! (terkunci ~6.4e-12) — loop training layak dipercaya.')
print('   Di produksi: torch.autograd.gradcheck memeriksa hal yang sama.')


## Bagian 5 — Kuantisasi & Kompresi

Kuantisasi simetris per-tensor: `skala = max|w| / (2^(bit-1) - 1)`,
`q = clamp(round(w/skala))`, dekuanti `w' = q × skala`. Ukuran GB turun
**linear** terhadap bit — tapi kualitas **tidak linear**; itulah kenapa
setiap level harus diukur.


In [ ]:
def kuantisasi_simulasi(bobot, bit):
    """Kuantisasi simetris: skala = max|w| / (2^(bit-1) - 1);
    q = clamp(round(w/skala)); dekuanti = q*skala. Return list BARU.
    bit >= 32 → salinan identik; skala 0 (semua nol) → salinan identik."""
    # TODO
    raise NotImplementedError


def laju_kompresi(bit):
    """FP32 → bit: rasio ukuran = 32/bit."""
    # TODO
    raise NotImplementedError


def ukuran_model_gb(param_miliar, bit):
    """Ukuran bobot (GB) = param × bit / 8 → GB (1e9)."""
    # TODO
    raise NotImplementedError


def bandingkan_varian(varian=None):
    """VARIAN_KUANTISASI → tabel dict {nama, bit, ukuran_gb (7B, round 2),
    kompresi (round 1), skor}."""
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 5.1 — kuantisasi simetris
w = [0.6, -0.3, 0.15, 0.05, -0.5]
q8 = kuantisasi_simulasi(w, 8)
q4 = kuantisasi_simulasi(w, 4)
q2 = kuantisasi_simulasi(w, 2)
print('bit=8:', [round(v, 6) for v in q8])
print('bit=4:', [round(v, 6) for v in q4])
print('bit=2:', [round(v, 6) for v in q2])
assert [round(v, 6) for v in q8] == [0.6, -0.302362, 0.151181, 0.051969, -0.500787]
assert [round(v, 6) for v in q4] == [0.6, -0.342857, 0.171429, 0.085714, -0.514286]
assert [round(v, 6) for v in q2] == [0.6, 0.0, 0.0, 0.0, -0.6]  # kekasaran ekstrem
assert kuantisasi_simulasi(w, 32) == list(w)                    # bit >= 32 identik
assert kuantisasi_simulasi([0.0, 0.0], 8) == [0.0, 0.0]         # skala 0 aman
assert w == [0.6, -0.3, 0.15, 0.05, -0.5]                       # input tak berubah
print('✅ Latihan 5.1 benar! bit kecil = level sedikit = detail hilang (lihat bit=2).')


In [ ]:
# ✅ Cek latihan 5.2 — ukuran 7B & tabel varian
assert laju_kompresi(16) == 2.0 and laju_kompresi(8) == 4.0 and laju_kompresi(4) == 8.0
assert ukuran_model_gb(7.0, 16) == 14.0
tabel = bandingkan_varian()
print(f"{'nama':>4} | {'bit':>3} | {'ukuran':>6} | {'kompresi':>8} | skor")
for t in tabel:
    print(f"{t['nama']:>4} | {t['bit']:>3} | {t['ukuran_gb']:>5} GB | {t['kompresi']:>7}x | {t['skor']}")
assert [(t['ukuran_gb'], t['kompresi'], t['skor']) for t in tabel] == [(14.0, 2.0, 0.82), (7.0, 4.0, 0.81), (3.5, 8.0, 0.78)]
print('✅ Latihan 5.2 benar! Ukuran turun linear (14→7→3.5), skor TIDAK (0.82→0.81→0.78).')
print('   fp16→int8 hampir gratis; int8→q4 mulai terasa. Ukur per task, jangan asumsi.')


## Bagian 6 — Trainer SGD + Early Stopping

Siklus Bab 4 tetap berlaku: forward → loss → gradien → update → val. Dua
kontrak yang sering salah:

1. **loss dicatat dari p SEBELUM update** tiap contoh (bukan loss setelahnya);
2. **restore bobot TERBAIK** (val loss minimum), bukan bobot terakhir —
   early stopping tanpa restore = berhenti di waktu yang tepat dengan bobot
   yang salah.


In [ ]:
def epoch_satu(contoh, w, b, lr):
    """Satu epoch SGD (urutan dataset, tanpa shuffle): update per contoh.

    Return (w_baru, b_baru, loss_rata) — loss dicatat dari p SEBELUM update
    tiap contoh.
    """
    # TODO
    raise NotImplementedError


def latih(train, val, lr=0.5, epochs=30, sabar=4, seed=13):
    """Loop SFT + early stopping.

    - val loss & acc dihitung SETIAP epoch;
    - best = epoch dengan val loss minimum (1-based);
    - berhenti bila val tak membaik `sabar` epoch berturut-turut;
    - restore bobot TERBAIK (bukan bobot terakhir!).

    Return dict: w, b, riwayat {'train_loss','val_loss','val_acc'},
    best_epoch, berhenti_dini, epochs_dijalankan.
    """
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 6.1 — satu epoch, angka terkunci
w0, b0 = inisialisasi()
loss_awal = loss_batch(data['train'], w0, b0)
w1, b1, tr = epoch_satu(data['train'], w0, b0, lr=0.5)
print(f'loss sebelum: {loss_awal:.4f} | train_loss epoch-1 (sebelum update): {tr:.6f}')
print(f'b setelah 1 epoch: {b1:.6f}')
assert abs(tr - 0.739636) < 1e-4, 'loss harus dicatat SEBELUM update tiap contoh'
assert abs(b1 - -0.114574) < 1e-4
assert loss_batch(data['train'], w1, b1) < loss_awal   # turun setelah 1 epoch
print('✅ Latihan 6.1 benar! train_loss 0.739636 = rata loss dari p sebelum update.')


In [ ]:
# ✅ Cek latihan 6.2 — data bersih: konvergen, tidak berhenti dini
h = latih(data['train'], data['val'], lr=0.5, epochs=60, sabar=6)
r = h['riwayat']
print(f"train_loss: {r['train_loss'][0]:.4f} → {r['train_loss'][-1]:.4f}"
      f" | val_acc akhir {r['val_acc'][-1]:.3f} | dini {h['berhenti_dini']} | best ep {h['best_epoch']}")
assert not h['berhenti_dini'] and h['epochs_dijalankan'] == 60
assert r['val_acc'][-1] == 1.0
assert abs(r['train_loss'][0] - 0.739636) < 1e-4
assert abs(r['train_loss'][-1] - 0.0872) < 1e-3
assert h['best_epoch'] == 60   # val loss masih membaik sampai akhir
print('✅ Latihan 6.2 benar! Data bersih: loss 0.7396 → 0.0872, val acc 1.0.')


In [ ]:
# ✅ Cek latihan 6.3 — data kontradiktif: OVERFIT, berhenti dini, restore BEST
train_balik = [(x, (1.0 - y) if i in (6, 14) else y) for i, (x, y) in enumerate(data['train'])]
h2 = latih(train_balik, data['val'], lr=3.0, epochs=100, sabar=5)
r2 = h2['riwayat']
print(f"berhenti dini {h2['berhenti_dini']} | dijalankan {h2['epochs_dijalankan']}"
      f" | BEST epoch {h2['best_epoch']}")
print(f"val_loss best {min(r2['val_loss']):.4f} vs akhir {r2['val_loss'][-1]:.4f}"
      f" | acc di bobot akhir {r2['val_acc'][-1]:.3f}")
assert h2['berhenti_dini'] and h2['epochs_dijalankan'] == 66
assert h2['best_epoch'] == 61
assert abs(min(r2['val_loss']) - 0.4184) < 1e-3
assert abs(r2['val_loss'][-1] - 0.4188) < 1e-3      # memburuk setelah best
assert r2['val_acc'][h2['best_epoch'] - 1] == 0.75
# restore best: model yang dikembalikan = bobot best (bukan bobot epoch 66)
vl_restored = loss_batch(data['val'], h2['w'], h2['b'])
assert abs(vl_restored - min(r2['val_loss'])) < 1e-9, 'harus me-restore bobot TERBAIK'
print('✅ Latihan 6.3 benar! Best epoch 61 ≠ epoch terakhir 66 — bobot TERBAIK yang',
      'dipulihkan. Training lebih lama BUKAN selalu lebih baik.')


## Bagian 7 — Biaya: API vs Lokal adalah Fungsi Volume

Open-weight lokal membuat biaya marginal token = 0, tapi GPU jadi biaya tetap.
Keputusan = bandingkan proyeksi API bulanan vs sewa GPU — **angka, bukan tren**.


In [ ]:
def biaya_api(token_in, token_out, model='api_mini'):
    """USD = in/1e6*harga_in + out/1e6*harga_out ('lokal' = 0.0)."""
    # TODO
    raise NotImplementedError


def biaya_bulanan(permintaan_per_hari, token_in, token_out, model='api_mini', hari=30):
    """biaya_api × permintaan_per_hari × hari."""
    # TODO
    raise NotImplementedError


def token_estimasi(teks):
    """ceil(len(teks)/4) — cukup untuk proyeksi, bukan tagihan."""
    # TODO
    raise NotImplementedError


def keputusan_deployment(permintaan_per_hari, token_in, token_out,
                         biaya_gpu_per_bulan=60.0, hari=30):
    """biaya API bulanan vs GPU: api > gpu → 'lokal', selain itu 'api'.
    Return {'biaya_api_bulanan', 'biaya_lokal_bulanan', 'keputusan'}."""
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 7.1 — tarif & proyeksi bulanan
for m in ('api_besar', 'api_mini', 'lokal'):
    print(f'{m:>9}: ${biaya_api(1000, 300, m):.6f} per permintaan')
assert abs(biaya_api(1000, 300, 'api_besar') - 0.0055) < 1e-9
assert abs(biaya_api(1000, 300, 'api_mini') - 0.00033) < 1e-9
assert biaya_api(1000, 300, 'lokal') == 0.0
assert abs(biaya_bulanan(2000, 1000, 300, 'api_mini') - 19.8) < 1e-9
assert abs(biaya_bulanan(10000, 1000, 300, 'api_mini') - 99.0) < 1e-9
assert token_estimasi('abcd') == 1
print('bulanan 2.000 req/hari: $19.8 | 10.000 req/hari: $99.0 — GPU $60/bln')
print('✅ Latihan 7.1 benar!')


In [ ]:
# ✅ Cek latihan 7.2 — keputusan & breakeven
k2 = keputusan_deployment(2000, 1000, 300)
k10 = keputusan_deployment(10000, 1000, 300)
print('2.000 req/hari :', k2['keputusan'], f"(${k2['biaya_api_bulanan']:.2f}/bln)")
print('10.000 req/hari:', k10['keputusan'], f"(${k10['biaya_api_bulanan']:.2f}/bln)")
assert k2['keputusan'] == 'api' and k10['keputusan'] == 'lokal'
breakeven = 60.0 / 30.0 / biaya_api(1000, 300, 'api_mini')
print('breakeven ≈', round(breakeven), 'req/hari')
assert 6060 < breakeven < 6062   # (60/30) / 0.00033 ≈ 6.060,6
print('✅ Latihan 7.2 benar! Di bawah ~6.061 req/hari → API; di atas → GPU lokal.')
print('   Penggeser ambang nyata: harga token, privasi, uptime (Bab 16), utilisasi GPU.')


## Bagian 8 — Evaluasi Jujur + Kebijakan OOD (abstain, Bab 12)

Fine-tune tanpa evaluasi = \"rasanya lebih pintar\". Di sini: akurasi val
sebelum vs sesudah, plus **kebijakan OOD** — model menghitung
`yakin = |p - ambang| × 2`; bila yakin < skor_min → jangan tampilkan label.


In [ ]:
def prediksi(teks, w, b, tok=None, ambang=0.5):
    """teks → {'label' (positif bila p>=ambang), 'skor' (p),
    'yakin' = |p - ambang| * 2}."""
    # TODO
    raise NotImplementedError


def evaluasi_sebelum_sesudah(val, w_sebelum, b_sebelum, w_sesudah, b_sesudah):
    """{'sebelum': acc, 'sesudah': acc, 'delta': sesudah - sebelum}."""
    # TODO
    raise NotImplementedError


def kebijakan_ood(hasil_pred, skor_min=0.6, jawaban_aman='kurang yakin, mohon periksa'):
    """yakin < skor_min → {'tampilkan': False, 'aksi': jawaban_aman};
    selain itu {'tampilkan': True, 'aksi': 'tampilkan label'}.
    Jangan mengubah hasil_pred — kembalikan dict baru (merge)."""
    # TODO
    raise NotImplementedError


def laporan_eval(val, w, b, ood_hasil=None, skor_min=0.6):
    """{'akurasi': acc(val), 'salah': [{'p','y'} untuk contoh salah],
    'ood': kebijakan_ood(ood_hasil) bila diberikan, selain itu None}."""
    # TODO
    raise NotImplementedError


In [ ]:
# ✅ Cek latihan 8.1 — sebelum/sesudah + delta
w0, b0 = inisialisasi()
ev = evaluasi_sebelum_sesudah(data['val'], w0, b0, h['w'], h['b'])
print('sebelum:', ev['sebelum'], '| sesudah:', ev['sesudah'], '| delta:', ev['delta'])
assert ev == {'sebelum': 0.75, 'sesudah': 1.0, 'delta': 0.25}
print('✅ Latihan 8.1 benar! Delta +0.25 — angka, bukan \'rasanya lebih pintar\'.')


In [ ]:
# ✅ Cek latihan 8.2 — kebijakan OOD: model boleh bilang 'kurang yakin'
p_ood = prediksi(DATA_OOD['teks'], h['w'], h['b'])
print(f"OOD {DATA_OOD['teks']!r} → skor {p_ood['skor']:.4f} | yakin {p_ood['yakin']:.4f}")
assert abs(p_ood['skor'] - 0.4988) < 1e-3 and abs(p_ood['yakin'] - 0.0024) < 1e-3
kep = kebijakan_ood(p_ood)
print('keputusan:', kep['tampilkan'], '|', kep['aksi'])
assert kep['tampilkan'] is False and kep['aksi'] == 'kurang yakin, mohon periksa'
assert kep != p_ood and 'tampilkan' not in p_ood    # dict BARU, input utuh

# contoh in-distribution: yakin tinggi → tampilkan
p_pos = prediksi('Barang bagus, pengiriman cepat', h['w'], h['b'])
kep_pos = kebijakan_ood(p_pos)
print(f"in-dist {p_pos['label']} (yakin {p_pos['yakin']:.2f}) → tampilkan {kep_pos['tampilkan']}")
assert kep_pos['tampilkan'] is True and kep_pos['aksi'] == 'tampilkan label'
print('✅ Latihan 8.2 benar! OOD tertahan (abstain, Bab 12); label yakin dilewatkan.')


In [ ]:
# ✅ Cek latihan 8.3 — laporan eval
lap = laporan_eval(data['val'], h['w'], h['b'], ood_hasil=p_ood)
print('akurasi:', lap['akurasi'], '| salah:', lap['salah'], '| ood:', lap['ood']['tampilkan'])
assert lap['akurasi'] == 1.0 and lap['salah'] == []
assert lap['ood']['tampilkan'] is False
lap2 = laporan_eval(data['val'], w0, b0)   # tanpa ood → None
assert lap2['ood'] is None and len(lap2['salah']) == 2
print('✅ Latihan 8.3 benar! Laporan = akurasi + contoh salah + keputusan OOD.')


## Bagian 9 — Ringkasan: Satu Siklus, Tujuh Angka

Angka dari bagian sebelumnya disatukan — inilah yang dilaporkan saat pipeline
fine-tuning ini berubah (Bab 15).


In [ ]:
# ✅ Cek latihan 9.1 — ringkasan terkunci
print('=' * 64)
print('RINGKASAN: siklus fine-tuning tanpa GPU (SEED 13)')
print('=' * 64)
print('1. dataset      : 24 contoh (16 train / 8 val) | SFT 8 baris (5/3)')
print('2. split & ov   : (5, 3) | oversample 6/6 = 12 baris, urutan asli')
print('3. LoRA         : 4096² r16 = 131.072 param = 0.78% dari full')
print('4. gradient chk : max err', gradient_check(), '(< 1e-6)')
print('5. kuantisasi   : 14 GB (0.82) → 7 GB (0.81) → 3.5 GB (0.78)')
print('6. trainer      : bersih 0.7396→0.0872 acc 1.0 | overfit best ep 61, stop 66')
print('7. biaya        : breakeven ~6.061 req/hari | eval 0.75→1.0 | OOD tertahan')
print()
print('Semua angka terkunci karena tokenizer, model, dan gradien semuanya')
print('deterministik. Itulah inti bab ini: fine-tuning yang serius diukur —')
print('sebelum/sesudah, OOD, dan keputusan biaya — bukan \'rasanya lebih pintar\'.')


### Koneksi ke bab lain

- **Bab 4:** loss, overfitting, val split, early stopping — siklusnya identik,
  skalanya yang berbeda.
- **Bab 11:** tarif per token & keputusan berbasis angka; open-weight lokal =
  `base_url` baru, kode orkestrasi tetap jalan.
- **Bab 12:** abstain saat tidak yakin → di sini jadi kebijakan OOD
  (`yakin < 0.6 → jangan tampilkan`).
- **Bab 15 (Evaluasi):** eval sebelum/sesudah = versi paling sederhana dari
  regression testing; LLM nyata perlu eval set + judge rubrik.
- **Bab 16 (Deployment):** GPU lokal = uptime jadi tanggunganmu; kuantisasi =
  tuas VRAM/latensi.

**Pertanyaan analisis** (jawab di sini dengan merujuk angka eksperimenmu):

1. Rasio param LoRA 0.78% — mana yang lebih murah di produksi: 1 adapter besar
   per klien atau 100 adapter kecil? Kapan rank perlu dinaikkan?
2. Early stopping berhenti di epoch 66 dari 100, best epoch 61. Apa yang
   terjadi pada val loss setelah epoch 61, dan mengapa bobot TERBAIK (bukan
   terakhir) yang dipulihkan?
3. Oversampling menduplikasi data minoritas (5/3 → 6/6). Risiko apa yang
   muncul bila duplikasi terlalu banyak, dan alternatif apa selain duplikasi?
4. Kuantisasi Q4 memangkas 14 GB → 3.5 GB dengan skor 0.82 → 0.78. Untuk kasus
   apa kompromi itu layak, dan kapan TIDAK layak?
5. Breakeven API↔lokal ≈ 6.061 req/hari (asumsi modul). Faktor nyata apa yang
   bisa menggeser ambang itu?
6. Klasifier ini tahu dia tidak yakin via satu angka `yakin`. LLM open-weight
   tidak punya satu angka 'yakin' — bagaimana mendeteksi ketidakpastian di LLM
   nyata, dan lapisan mana yang tetap wajib (Bab 12 & 15)?

Lanjut ke: **kuis** (`02_kuis_finetuning.ipynb`) lalu **project starter**
(`project-ftkit/`) — paket `ftkit` TDD dengan **44 test**.
